BAT Burnout Annotation Script — Groq Edition
=============================================
Runs a Groq-hosted LLM on each Reddit post using the BAT codebook prompt.
Outputs a CSV with BAT scores (EX, EMO, COG, MD), triage decision,
rationale, and agreement check against the original Label column.

Usage:
    python bat_annotator_groq.py
    python bat_annotator_groq.py --limit 5                        # test on 5 rows
    python bat_annotator_groq.py --model llama-3.1-8b-instant     # faster/cheaper
    python bat_annotator_groq.py --input data.csv --output out.csv

Recommended Groq models (free tier available at console.groq.com):
    llama-3.3-70b-versatile   — best quality,  ~$0.59/1M tokens  [DEFAULT]
    llama-3.1-70b-versatile   — good quality,  similar pricing
    llama-3.1-8b-instant      — fastest/cheapest, lower quality
    mixtral-8x7b-32768        — good for long posts (32k context)
    gemma2-9b-it              — lightweight option

In [40]:
!pip install groq

In [41]:
import groq as groq_sdk
import pandas as pd
import json
import time
import argparse
import os
import sys
import re

In [42]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [43]:
# parse argument
parser = argparse.ArgumentParser(description="BAT Burnout Annotator using Groq")
parser.add_argument("--input",   default="/content/drive/MyDrive/Reddit_research spring26/codebook/fewshot_examples.csv")
parser.add_argument("--output",  default="/content/drive/MyDrive/Reddit_research spring26/codebook/outputs/bat_scored_results_groq.csv")
parser.add_argument("--limit",   type=int,   default=None,  help="Only process first N rows (for testing)")
parser.add_argument("--model",   default="llama-3.3-70b-versatile", help="Groq model name")
parser.add_argument("--delay",   type=float, default=0.5,   help="Seconds between API calls (Groq is fast, 0.5 is fine)")
parser.add_argument("--temp",    type=float, default=0.0,   help="Temperature (0 = deterministic, recommended for annotation)")
args = parser.parse_args(args=[])

In [44]:
print("working")

working


In [45]:
# codebook system prompt
SYSTEM_PROMPT = """You are an expert annotation assistant applying the Burnout Assessment Tool (BAT) codebook to Reddit posts from cybersecurity communities.

## YOUR TASK
For each Reddit post, you will:
1. Decide if the post is IN-SCOPE or OUT-OF-SCOPE (N/A)
2. If IN-SCOPE, assign a 1-5 score for each of the four BAT categories
3. Provide a brief rationale for each decision

---

## STEP 1 — TRIAGE: IS THE POST IN-SCOPE?

Code as IN-SCOPE if the post contains first-person expression of:
- Work-related exhaustion, depletion, or inability to recover
- Emotional dysregulation or distress connected to work
- Cognitive difficulties (focus, memory, decisions) at work
- Withdrawal, detachment, or loss of meaning in work
- Related work stressors: overload, toxic environment, job loss, imposter syndrome, sleep disruption, anxiety tied to work

Code as OUT-OF-SCOPE (N/A) if the post is one of:
- N/A_AWARENESS: Talks about burnout generally without personal symptom expression
- N/A_INFORMATIONAL: Survey invitations, research requests, resource links
- N/A_EXTERNAL_STRESSOR: Industry stressor (layoffs, AI, job market) without personal impairment
- N/A_CAREER_CURIOSITY: Hypothetical questions about roles/career with no personal distress
- N/A_ABOUT_OTHERS: Describes another person's burnout, not the poster's own

---

## STEP 2 — BAT SCORING (only if IN-SCOPE)

Score each category independently from 1 to 5:
1 = Never / Not present — No evidence in the post
2 = Rarely / Vague — Passing or hedged mention; less than 60% confident it is present
3 = Sometimes / Moderate — Clear but mild; present but not dominant
4 = Often / Strong — Explicit, sustained; clearly a significant theme
5 = Always / Severe — Dominates the post; intense, repeated, extreme language

If OUT-OF-SCOPE, assign score 1 to all four categories.

---

## CATEGORY DEFINITIONS & KEY SIGNALS

### EX — Exhaustion
BAT definition: Severe loss of energy (physical + mental). Lack of energy to start work, feeling used-up, tiring quickly with minimal effort, inability to relax after work.
Core question: Does the poster describe running out of energy because of work and does rest fail to restore them?
Score HIGH (4-5) for: "drained", "nothing left", "can't decompress", "exhausted before the day starts", "sleep doesn't help", "running on empty", "physically broken"
Do NOT score high if: only general stress without energy loss; tiredness is passing; post is purely hypothetical

### EMO — Emotional Impairment
BAT definition: Intense emotional reactions, feeling overwhelmed. Frustrated/angry at work, irritability, overreacting, upset/sad without knowing why, unable to control emotions.
Core question: Does the poster describe emotions that feel out of control, disproportionate, or unpredictable at work?
Score HIGH (4-5) for: "I snap at colleagues", "can't control it", "crying without knowing why", "I don't recognise myself", persistent anger/frustration
Do NOT score high if: frustration is clearly tied to one specific event and proportionate; post is hypothetical

### COG — Cognitive Impairment
BAT definition: Memory problems, attention/concentration deficits. Difficulty thinking clearly, forgetful, absent-minded, indecision, trouble focusing.
Core question: Does the poster describe thinking, memory, focus, or decision-making as impaired beyond a new role or single bad day?
Score HIGH (4-5) for: "brain fog", "keep forgetting things", "can't concentrate", "second-guess everything", "forgot a procedure I have done for years", explicit decision-making failure
Do NOT score high if: person is new to the role and learning normally; single off-day; post is hypothetical

### MD — Mental Distance
BAT definition: Psychological withdrawal from work. Strong reluctance/aversion to work, avoids colleagues, indifference, cynicism, no enthusiasm, functions on autopilot.
Core question: Does the poster describe withdrawal, indifference, cynicism, or persistent loss of meaning toward their work?
Score HIGH (4-5) for: "don't care anymore", "what's the point", "going through the motions", "I used to love this", "avoid my team", persistent cynical tone
Do NOT score high if: single bad day, one-off complaint, or purely hypothetical question

---

## BOUNDARY NOTES

- EX vs. MD: "No energy" = EX. "No interest/don't care" = MD. Both can co-exist.
- EMO vs. MD: "Emotions too intense" = EMO. "Emotions flat/absent/numb" = MD. Rarely both high simultaneously.
- COG vs. MD (autopilot): "Can't process/think" = COG. "Don't care/checked out" = MD. Both can co-exist.
- Situational frustration is NOT EMO: Anger about one specific proportionate event = score 1-2, not 4-5.
- Hypothetical posts: Questions like "Is SOC stressful?" with no personal distress = N/A_CAREER_CURIOSITY.
- Sarcasm/dark humour counts as MD if persistent throughout post.
- The word "burnout" alone without symptom expression = score 2 maximum.

---

## CONFIDENCE RULE
- Greater than 70% confident symptom present: score 3, 4, or 5
- 50-70% confident (ambiguous): score 2
- Less than 50% confident: score 1

---

## OUTPUT FORMAT — CRITICAL INSTRUCTIONS
You MUST respond with a single valid JSON object and nothing else.
Do NOT include any text before or after the JSON.
Do NOT use markdown code fences.
Do NOT add any explanation outside the JSON.

The JSON must follow this exact structure:
{
  "triage": "IN_SCOPE",
  "na_subtype": null,
  "topic_type": "T1_PersonalBurnout",
  "scores": {
    "EX": 3,
    "EMO": 2,
    "COG": 1,
    "MD": 4
  },
  "rationale": {
    "triage": "Explanation here.",
    "EX": "Explanation here.",
    "EMO": "Explanation here.",
    "COG": "Explanation here.",
    "MD": "Explanation here."
  },
  "dominant_category": "MD",
  "confidence": "HIGH"
}

Valid values:
- triage: "IN_SCOPE" or "OUT_OF_SCOPE"
- na_subtype: null, "N/A_AWARENESS", "N/A_INFORMATIONAL", "N/A_EXTERNAL_STRESSOR", "N/A_CAREER_CURIOSITY", or "N/A_ABOUT_OTHERS"
- topic_type: "T1_PersonalBurnout", "T2_GeneralCyberBurnout", "T3_SOCAnalystStress", "T4_WorkLifeBalance", "T5_MentalHealthAwareness", or "T6_JobMarketCareer"
- scores EX/EMO/COG/MD: integers 1, 2, 3, 4, or 5
- dominant_category: "EX", "EMO", "COG", "MD", or "NONE"
- confidence: "HIGH", "MEDIUM", or "LOW"
"""

In [46]:
# user prompt
def build_user_prompt(post_text: str) -> str:
    return f"""Annotate this Reddit post using the BAT codebook. Reply with JSON only.

POST:
{post_text.strip()}"""

In [47]:
# json extraction for LLM's output
def extract_json(raw: str) -> dict:
    raw = raw.strip()
    # strip markdown code fences if present
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)
    # find the outermost JSON object
    start = raw.find("{")
    end   = raw.rfind("}") + 1
    if start != -1 and end > start:
        raw = raw[start:end]
    return json.loads(raw)

In [48]:
#empty or error result
def empty_result(error_msg: str) -> dict:
    return {
        "triage": "ERROR",
        "na_subtype": None,
        "topic_type": None,
        "scores": {"EX": None, "EMO": None, "COG": None, "MD": None},
        "rationale": {"triage": error_msg, "EX": "", "EMO": "", "COG": "", "MD": ""},
        "dominant_category": None,
        "confidence": None,
        "error": error_msg,
        "raw_response": "",
    }

In [49]:
#Annotation Call
def annotate_post(client, post_id: str, post_text: str) -> dict:
    try:
        response = client.chat.completions.create(
            model=args.model,
            temperature=args.temp,
            max_tokens=2000,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": build_user_prompt(post_text)},
            ],
        )
        raw = response.choices[0].message.content
        result = extract_json(raw)
        result["raw_response"] = raw
        return result

    except json.JSONDecodeError as e:
        raw = response.choices[0].message.content if "response" in dir() else ""
        print(f"  [!] JSON parse error for {post_id}: {e}")
        print(f"      Raw output: {raw[:200]}")
        r = empty_result(f"JSON parse error: {e}")
        r["raw_response"] = raw
        return r
    except Exception as e:
        print(f"  [!] Error for {post_id}: {type(e).__name__}: {e}")
        return empty_result(f"{type(e).__name__}: {e}")

In [50]:
# now write the results in csv
def flatten(post_id:str, original_label:int, label_reason:str, result:dict)-> dict:
  scores     = result.get("scores", {})
  rationale  = result.get("rationale", {})
  score_vals = [scores.get(k) for k in ["EX", "EMO", "COG", "MD"]]
  total      = sum(v for v in score_vals if v is not None) if any(v is not None for v in score_vals) else None
  llm_label  = 1 if result.get("triage") == "IN_SCOPE" else (0 if result.get("triage") == "OUT_OF_SCOPE" else None)
  match      = (llm_label == original_label) if llm_label is not None else None

  return {
      "post_id":             post_id,
      "original_label":      original_label,
      "label_reason":        label_reason,
      "llm_triage":          result.get("triage"),
      "llm_label":           llm_label,
      "label_match":         match,
      "na_subtype":          result.get("na_subtype"),
      "topic_type":          result.get("topic_type"),
      "EX_score":            scores.get("EX"),
      "EMO_score":           scores.get("EMO"),
      "COG_score":           scores.get("COG"),
      "MD_score":            scores.get("MD"),
      "total_burnout_score": total,
      "dominant_category":   result.get("dominant_category"),
      "confidence":          result.get("confidence"),
      "rationale_triage":    rationale.get("triage", ""),
      "rationale_EX":        rationale.get("EX", ""),
      "rationale_EMO":       rationale.get("EMO", ""),
      "rationale_COG":       rationale.get("COG", ""),
      "rationale_MD":        rationale.get("MD", ""),
      "error":               result.get("error", ""),
      "raw_response":        result.get("raw_response", ""),
  }

In [51]:
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

In [52]:
# Main Function
def main():
  print("=" * 60)
  print("BAT Burnout Annotator — Groq Edition")
  print(f"Model  : {args.model}")
  print(f"Temp   : {args.temp}")
  print(f"Input  : {args.input}")
  print(f"Output : {args.output}")
  print("=" * 60)

  df = pd.read_csv(args.input)
  if args.limit:
      df = df.head(args.limit)
      print(f"[TEST MODE] Processing first {args.limit} rows only\n")

  client  = groq_sdk.Groq(api_key=GROQ_API_KEY)   # reads GROQ_API_KEY from environment automatically
  results = []
  errors  = 0
  total_n = len(df)

  for idx, (_, row) in enumerate(df.iterrows(), start=1):
      post_id = str(row["id"])
      text    = str(row["text"])
      label   = int(row["Label"])
      reason  = str(row.get("label_reason", ""))
      preview = text[:80].replace("\n", " ")

      print(f"[{idx:>3}/{total_n}] {post_id}  |  L={label}  |  \"{preview}...\"")
#call the functions
      result  = annotate_post(client, post_id, text)
      row_out = flatten(post_id, label, reason, result)
      results.append(row_out)

      triage = result.get("triage", "ERROR")
      scores = result.get("scores", {})
      match_sym = "✓" if row_out["label_match"] else ("✗" if row_out["label_match"] is False else "?")
      dom    = result.get("dominant_category", "-")
      conf   = result.get("confidence", "-")
      print(f"         {triage}  |  EX={scores.get('EX')} EMO={scores.get('EMO')} "
            f"COG={scores.get('COG')} MD={scores.get('MD')}  "
            f"|  dominant={dom}  conf={conf}  |  label={match_sym}")

      if result.get("error"):
          errors += 1

      if args.delay > 0:
          time.sleep(args.delay)

  # save the results
  out_df = pd.DataFrame(results)
  # drop raw_response from saved CSV to keep it readable (keep for debug)
  save_cols = [c for c in out_df.columns if c != "raw_response"]
  out_df[save_cols].to_csv(args.output, index=False)
  print(f"\n Results saved → {args.output}")




  # Summary Print
  print("\n" + "=" * 60)
  print("SUMMARY")
  print("=" * 60)

  n          = len(out_df)
  in_n       = (out_df["llm_triage"] == "IN_SCOPE").sum()
  out_n      = (out_df["llm_triage"] == "OUT_OF_SCOPE").sum()
  err_n      = errors
  matches    = out_df["label_match"].sum()
  match_pct  = matches / n * 100 if n > 0 else 0

  print(f"Total processed   : {n}")
  print(f"IN_SCOPE          : {in_n}  ({in_n/n*100:.1f}%)")
  print(f"OUT_OF_SCOPE      : {out_n}  ({out_n/n*100:.1f}%)")
  print(f"Label agreement   : {int(matches)}/{n}  ({match_pct:.1f}%)")
  print(f"Errors            : {err_n}")

  # Score distributions for in-scope posts
  in_df = out_df[out_df["llm_triage"] == "IN_SCOPE"]
  if len(in_df) > 0:
      print(f"\nAvg BAT scores (in-scope posts, n={len(in_df)}):")
      for cat in ["EX", "EMO", "COG", "MD"]:
          vals = in_df[f"{cat}_score"].dropna()
          if len(vals):
              dist = vals.value_counts().sort_index().to_dict()
              print(f"  {cat}: mean={vals.mean():.2f}  "
                    f"median={vals.median():.1f}  dist={dist}")

  # Dominant category
  dom_counts = in_df["dominant_category"].value_counts()
  if len(dom_counts):
      print(f"\nDominant category (in-scope):")
      for cat, cnt in dom_counts.items():
          print(f"  {cat}: {cnt}")

  # Confidence breakdown
  conf_counts = out_df["confidence"].value_counts()
  if len(conf_counts):
      print(f"\nConfidence breakdown:")
      for lvl, cnt in conf_counts.items():
          print(f"  {lvl}: {cnt}")

  # Triage vs label confusion matrix
  print(f"\nTriage vs. Original Label (0=out-of-scope, 1=in-scope):")
  confusion = pd.crosstab(
      out_df["llm_triage"].fillna("ERROR"),
      out_df["original_label"],
      rownames=["LLM →"],
      colnames=["Original →"],
  )
  print(confusion.to_string())

  # N/A sub-type breakdown
  na_rows = out_df[out_df["llm_triage"] == "OUT_OF_SCOPE"]
  if len(na_rows):
      print(f"\nN/A sub-types:")
      for sub, cnt in na_rows["na_subtype"].value_counts().items():
          print(f"  {sub}: {cnt}")

  # Posts where LLM disagrees with original label
  disagree = out_df[out_df["label_match"] == False]
  if len(disagree):
      print(f"\nDisagreements ({len(disagree)} posts) — review these:")
      for _, r in disagree.iterrows():
          print(f"  {r['post_id']}  original={r['original_label']}  "
                f"llm={r['llm_label']}  triage={r['llm_triage']}")
          print(f"    reason: {r['rationale_triage'][:100]}")

  print("\nDone.")


if __name__ == "__main__":
  main()


BAT Burnout Annotator — Groq Edition
Model  : llama-3.3-70b-versatile
Temp   : 0.0
Input  : /content/drive/MyDrive/Reddit_research spring26/codebook/fewshot_examples.csv
Output : /content/drive/MyDrive/Reddit_research spring26/codebook/outputs/bat_scored_results_groq.csv
[  1/40] 1ga6hmb  |  L=1  |  "What Do You Like and Dislike About Your Job in Cybersecurity? Here’s My Experien..."
         IN_SCOPE  |  EX=2 EMO=3 COG=1 MD=4  |  dominant=MD  conf=MEDIUM  |  label=✓
[  2/40] 1fdw53a  |  L=1  |  "Work/life balance in cybersecurity.. is there really such a thing? Hey folks.. I..."
         IN_SCOPE  |  EX=4 EMO=2 COG=1 MD=2  |  dominant=EX  conf=HIGH  |  label=✓
[  3/40] 1gdqpjt  |  L=1  |  "Frustrated with being an intern, despite feeling like a full-timer. I'm going to..."
         IN_SCOPE  |  EX=2 EMO=4 COG=1 MD=3  |  dominant=EMO  conf=HIGH  |  label=✓
[  4/40] 1fgs08n  |  L=0  |  "Roast My resume..."
         OUT_OF_SCOPE  |  EX=1 EMO=1 COG=1 MD=1  |  dominant=NONE  conf=HIGH  |  